In [1]:
import numpy as np
import pandas as pd
import pydicom
import os
import scipy.ndimage
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from skimage import measure, morphology
from sklearn.preprocessing import normalize

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset




In [2]:
TRAIN_FOLDER = "../data/train"


def load_scan(path):
    try:
        slices = [pydicom.dcmread(path + os.sep + s) for s in os.listdir(path)]
        slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    except:
        files = os.listdir(path)
        files.sort()
        slices = [pydicom.dcmread(path + os.sep + s) for s in files]
    return slices


def get_pixels_hu(slices):
    image = np.stack([s.pixel_array for s in slices])
    image = image.astype(np.int16)
    try:
        image[image <= -2000] = 0
        for slice_number in range(len(slices)):
            intercept = slices[slice_number].RescaleIntercept
            slope = slices[slice_number].RescaleSlope
            if slope != 1:
                image[slice_number] = slope * image[slice_number].astype(np.float64)
                image[slice_number] = image[slice_number].astype(np.int16)
            image[slice_number] += np.int16(intercept)
    except:
        print("HU conversion Failed!!")
    return np.array(image, dtype=np.int16)


def resize_along_allaxis(
    slices, target_dimensionZ=30, target_dimensionY=100, target_dimensionX=100
):
    present_dimensionZ, present_dimensionY, present_dimensionX = slices.shape
    if (
        target_dimensionZ == present_dimensionZ
        and target_dimensionY == present_dimensionY
        and target_dimensionX == present_dimensionX
    ):
        return slices
    zoom_factorZ = float(target_dimensionZ) / float(present_dimensionZ)
    zoom_factorY = float(target_dimensionY) / float(present_dimensionY)
    zoom_factorX = float(target_dimensionX) / float(present_dimensionX)
    resize_image = scipy.ndimage.zoom(
        slices, [zoom_factorZ, zoom_factorY, zoom_factorX], mode="nearest"
    )
    return resize_image


MIN_BOUND = -1000.0
MAX_BOUND = 400.0


def image_normalize(image):
    image = (image - MIN_BOUND) / (MAX_BOUND - MIN_BOUND)
    image[image > 1] = 1.0
    image[image < 0] = 0.0
    return image


def read_image(dir_name, patientid, Z=100, Y=200, X=200):
    path = dir_name + os.sep + patientid
    slices = load_scan(path)
    try:
        image_array = get_pixels_hu(slices)
        ctimage_resizedAll = resize_along_allaxis(
            image_array, target_dimensionX=X, target_dimensionY=Y, target_dimensionZ=Z
        )
        image = (image_normalize(ctimage_resizedAll) * 255.0).astype("uint8")
        return image
    except:
        print("PatientId:%s couldnt be converted" % patientid)
        return np.zeros((Z, Y, X), dtype="uint8")




In [3]:
def csv_preprocess(data):
    data["Healthy-FVC"] = round((data["FVC"] * 100) / data["Percent"])
    FE = ["Healthy-FVC"]
    COLS = ["Sex", "SmokingStatus"]
    for col in COLS:
        for mod in data[col].unique():
            FE.append(mod)
            data[mod] = (data[col] == mod).astype(int)
    data = data[["Patient", "Weeks", "FVC", "Age"] + FE]
    data = data.sort_values(["Patient", "Weeks"], ascending=True).reset_index(drop=True)
    rename_col = {"Weeks": "base_Weeks", "FVC": "base_FVC"}
    data = data.rename(columns=rename_col)
    npData = pd.DataFrame(
        columns=["Patient", "base_Weeks", "base_FVC", "Age"]
        + FE
        + [
            "Male",
            "Female",
            "Ex-smoker",
            "Never smoked",
            "Currently smokes",
            "Week",
            "actual_FVC",
        ]
    )
    for pid in data["Patient"].unique():
        weeks = data.loc[data["Patient"] == pid].base_Weeks
        fvc = data.loc[data["Patient"] == pid].base_FVC
        idx = data.loc[data["Patient"] == pid].index
        weeks = weeks.reset_index(drop=True)
        fvc = fvc.reset_index(drop=True)
        for k in range(len(weeks)):
            row = data.loc[idx[0]].copy()
            row["Week"] = weeks[k]
            row["actual_FVC"] = fvc[k]
            npData = pd.concat([npData, row.to_frame().T], ignore_index=True)
    npData = npData.fillna(0)
    return npData




In [4]:
class Flatten(nn.Module):
    def forward(self, input):
        return input.view(input.size(0), -1)


class ds_3d_conv(nn.Module):
    def __init__(self, nin, nout, kernel_size, padding, kernels_per_layer):
        super(ds_3d_conv, self).__init__()
        self.depthwise = nn.Conv3d(
            nin,
            nin * kernels_per_layer,
            kernel_size=kernel_size,
            padding=padding,
            groups=nin,
        )
        self.pointwise = nn.Conv3d(nin * kernels_per_layer, nout, kernel_size=1)

    def forward(self, x):
        out = self.depthwise(x)
        out = self.pointwise(out)
        return out


class SIGMA(nn.Module):
    def __init__(self):
        super(SIGMA, self).__init__()
        self.data_net1 = nn.Sequential(
            nn.Linear(42, 64), nn.ReLU(), nn.Linear(64, 118), nn.ReLU()
        )
        self.data_net2 = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 502), nn.ReLU()
        )
        self.data_net3 = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 118), nn.ReLU()
        )
        self.data_net4 = nn.Sequential(
            nn.Linear(780, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
            nn.ReLU(),
        )

    def forward(self, data_i, image_o):
        x = torch.cat((data_i, image_o), dim=-1)
        out1 = self.data_net1(x)
        out2 = torch.cat((data_i, out1), dim=-1)
        out2 = self.data_net2(out2)
        out3 = torch.cat((data_i, out2), dim=-1)
        out3 = self.data_net3(out3)
        out4 = torch.cat((x, out1, out2, out3), dim=-1)
        out = self.data_net4(out4)
        return out


class IMAGE(nn.Module):
    def __init__(
        self, channel_number=[32, 64, 128, 256, 256, 64], output_dim=16, dropout=True
    ):
        super(IMAGE, self).__init__()
        n_layer = len(channel_number)
        self.feature_extractor = nn.Sequential()
        for i in range(n_layer):
            in_channel = 1 if i == 0 else channel_number[i - 1]
            out_channel = channel_number[i]
            if i < n_layer - 1:
                self.feature_extractor.add_module(
                    "conv_%d" % i,
                    self.conv_layer(
                        in_channel,
                        out_channel,
                        maxpool=True,
                        kernel_size=3,
                        padding=1,
                        kernels_per_layer=1,
                    ),
                )
            else:
                self.feature_extractor.add_module(
                    "conv_%d" % i,
                    self.conv_layer(
                        in_channel,
                        out_channel,
                        maxpool=False,
                        kernel_size=1,
                        padding=0,
                        kernels_per_layer=1,
                    ),
                )
        self.classifier = nn.Sequential()
        if dropout:
            self.classifier.add_module("dropout", nn.Dropout(0.5))
        self.classifier.add_module(
            "conv_final",
            nn.Conv3d(channel_number[-1], output_dim, padding=0, kernel_size=1),
        )
        self.flat = nn.Sequential(
            Flatten(),
            nn.Linear(1728, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
        )

    @staticmethod
    def conv_layer(
        in_channel,
        out_channel,
        maxpool=True,
        kernel_size=3,
        padding=1,
        kernels_per_layer=1,
        maxpool_stride=2,
    ):
        if maxpool:
            layer = nn.Sequential(
                ds_3d_conv(
                    in_channel, out_channel, kernel_size, padding, kernels_per_layer
                ),
                nn.BatchNorm3d(out_channel),
                nn.MaxPool3d(2, stride=maxpool_stride),
                nn.ReLU(),
            )
        else:
            layer = nn.Sequential(
                ds_3d_conv(
                    in_channel, out_channel, kernel_size, padding, kernels_per_layer
                ),
                nn.BatchNorm3d(out_channel),
                nn.ReLU(),
            )
        return layer

    def forward(self, image_i):
        image_o = self.feature_extractor(image_i)
        image_o = self.classifier(image_o)
        image_o = self.flat(image_o)
        return image_o


class Combined_NET(nn.Module):
    def __init__(self):
        super(Combined_NET, self).__init__()
        self.image = IMAGE()
        self.data = SIGMA()

    def forward(self, image_i, data_i):
        image_o = self.image(image_i)
        data_o = self.data(data_i, image_o)
        return data_o




In [5]:
C1 = torch.tensor(70.0, dtype=torch.float32)
C2 = torch.tensor(1000.0, dtype=torch.float32)


def score(y_true, y_pred):
    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]
    sigma_clip = torch.max(sigma, C1)
    delta = torch.abs(y_true[:, 0] - fvc_pred)
    delta = torch.min(delta, C2)
    sq2 = torch.tensor(2.0).sqrt()
    metric = (delta / sigma_clip) * sq2 + (sigma_clip * sq2).log()
    return metric.mean()


def qloss(y_true, y_pred):
    qs = torch.tensor([0.25, 0.50, 0.75], device=y_pred.device, dtype=torch.float32)
    e = y_true - y_pred
    v = torch.max(qs * e, (qs - 1) * e)
    return v.mean()


def quartile_loss(y_true, y_pred, _lambda=0.65):
    return _lambda * qloss(y_true, y_pred) + (1 - _lambda) * score(y_true, y_pred)




In [6]:
def make_eval_data(npEval, model, device="cuda"):
    feature_cols = [
        "base_Weeks",
        "base_FVC",
        "Age",
        "Male",
        "Female",
        "Ex-smoker",
        "Never smoked",
        "Currently smokes",
        "Week",
        "Healthy-FVC",
    ]
    x_features = torch.tensor(npEval[feature_cols].values, dtype=torch.float32)
    x_patientids = npEval["Patient"].values

    dir_name = "../input/osic-pulmonary-fibrosis-progression/test"
    loaded_images = {
        pid: read_image(dir_name, pid, Z=100, Y=200, X=200)
        for pid in npEval["Patient"].unique()
    }

    if model is not None and torch.cuda.is_available() and device == "cuda":
        model.to("cuda")
    predictions = []

    for i, pid in enumerate(x_patientids):
        img = loaded_images[pid]
        x_image = (
            torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        )  # (1,1,Z,Y,X)
        x_feat = x_features[i].unsqueeze(0)  # (1, F)

        if model is not None and torch.cuda.is_available() and device == "cuda":
            x_image = x_image.cuda()
            x_feat = x_feat.cuda()

        if model is not None:
            with torch.no_grad():
                pred = model(x_image, x_feat)
        else:
            base_fvc = npEval.iloc[i]["base_FVC"]
            week_offset = npEval.iloc[i]["Week"] - npEval.iloc[i]["base_Weeks"]
            patient_id = npEval.iloc[i]["Patient"]
            patient_slope = patient_slopes.get(patient_id, avg_slope)
            pred_fvc = base_fvc + patient_slope * week_offset
            pred_fvc = max(pred_fvc, 0.0)

            # Expanded confidence interval (±200 ml) to increase σ and improve metric
            lower = max(pred_fvc - 200.0, 0.0)
            upper = pred_fvc + 200.0
            pred = torch.tensor([[lower, pred_fvc, upper]], dtype=torch.float32)

        predictions.append(pred.cpu().numpy()[0])

    preds = np.array(predictions)  # shape (N,3)
    npEval["FVC"] = preds[:, 1]
    npEval["Confidence"] = preds[:, 2] - preds[:, 0]
    return npEval




In [7]:
data_train = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
data_test_raw = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
submission = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

submission["Patient"] = submission["Patient_Week"].apply(lambda x: x.split("_")[0])
submission["Weeks"] = submission["Patient_Week"].apply(lambda x: int(x.split("_")[1]))
submission = submission.sort_values(["Patient", "Weeks"]).reset_index(drop=True)

merge = pd.merge(data_test_raw, submission, on="Patient", how="left")
merge = merge.drop(columns=["FVC_y"]).rename(
    columns={"FVC_x": "base_FVC", "Weeks_y": "Week", "Weeks_x": "base_Weeks"}
)

data = merge.copy()
data["Healthy-FVC"] = round((data["base_FVC"] * 100) / data["Percent"])
FE = ["Healthy-FVC"]
COLS = ["Sex", "SmokingStatus"]
for col in COLS:
    for mod in data[col].unique():
        FE.append(mod)
        data[mod] = (data[col] == mod).astype(int)
FE1 = ["Male", "Female", "Ex-smoker", "Never smoked", "Currently smokes"]
npData = pd.DataFrame(
    columns=["Patient", "base_Weeks", "base_FVC", "Age", "Healthy-FVC"] + FE1 + ["Week"]
)
npData = pd.concat([npData, data], ignore_index=True).fillna(0)

data_test = npData[
    [
        "Patient",
        "base_Weeks",
        "base_FVC",
        "Age",
        "Healthy-FVC",
        "Male",
        "Female",
        "Ex-smoker",
        "Never smoked",
        "Currently smokes",
        "Week",
    ]
].copy()

train_df = data_train.copy()
baseline = train_df.groupby("Patient")["Weeks"].idxmin()
baseline = (
    train_df.loc[baseline, ["Patient", "Weeks", "FVC"]]
    .rename(columns={"Weeks": "base_Week", "FVC": "base_FVC"})
    .reset_index(drop=True)
)

train_merged = pd.merge(train_df, baseline, on="Patient", how="left")
train_merged = train_merged[train_merged["Weeks"] != train_merged["base_Week"]]
train_merged["weeks_diff"] = train_merged["Weeks"] - train_merged["base_Week"]
train_merged["delta_fvc"] = train_merged["FVC"] - train_merged["base_FVC"]
train_merged = train_merged[train_merged["weeks_diff"] != 0]
train_merged["slope"] = train_merged["delta_fvc"] / train_merged["weeks_diff"]
avg_slope = train_merged["slope"].mean()
if np.isnan(avg_slope):
    avg_slope = 0.0

patient_slopes = train_merged.groupby("Patient")["slope"].mean().to_dict()
if not patient_slopes:
    patient_slopes = {}




/tmp/ipykernel_55/42989007.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  npData = pd.concat([npData, data], ignore_index=True).fillna(0)
/tmp/ipykernel_55/42989007.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  npData = pd.concat([npData, data], ignore_index=True).fillna(0)


In [8]:
model = Combined_NET()
try:
    state_path = (
        "../input/8other681/Epoch8_Score6.8104051457335615_Acc0.9310561826686985.pth"
    )
    model.load_state_dict(torch.load(state_path, map_location="cpu"))
except Exception as e:
    print(
        f"Pretrained model not found or failed to load ({e}); proceeding with fallback predictions."
    )
    model = None

test = make_eval_data(data_test.copy(), model)




Pretrained model not found or failed to load ([Errno 2] No such file or directory: '../input/8other681/Epoch8_Score6.8104051457335615_Acc0.9310561826686985.pth'); proceeding with fallback predictions.


In [9]:
for nid in test["Patient"].unique():
    idx = test[(test["Patient"] == nid) & (test["Week"] == test["base_Weeks"])].index
    if len(idx):
        test.at[idx[0], "FVC"] = test.at[idx[0], "base_FVC"]
        # Use the larger confidence (σ = 400 ml) for the baseline week
        test.at[idx[0], "Confidence"] = 400.0




In [10]:
test.loc[test["Confidence"] < 70, "Confidence"] = 70.0




In [11]:
submission["FVC"] = test["FVC"].values
submission["Confidence"] = test["Confidence"].values
submission.to_csv("submission.csv", index=False)